# 02 Conventional Methods

Merged notebook for modules 02a-02c in tutorial order.

In [ ]:
# @title ⚙️ Environment Setup (Run this first!)
!pip install pymatgen numpy matplotlib scipy -q
print("✅ Packages installed")

In [ ]:
# @title 📂 Load Tutorial Data
import os

REPO = "MRS_CH08_Tutorial"
REPO_URL = "https://github.com/Szymanski-Group/MRS_CH08_Tutorial.git"

if os.path.basename(os.getcwd()) == REPO:
    print("✅ Data already present")
elif os.path.exists(REPO):
    os.chdir(REPO)
    print("✅ Data already present")
else:
    !git clone {REPO_URL} -q
    os.chdir(REPO)
    print("✅ Data loaded successfully")


## 02a - Search Match

In [ ]:
import glob
from IPython.display import Image, display

# Inline tutorial script
from pathlib import Path
import csv

# For handling arrays
import numpy as np

# For plotting
import matplotlib.pyplot as plt

# For peak detection
from scipy.signal import find_peaks

# To load structures and compute XRD stick patterns
from pymatgen.core import Structure
from pymatgen.analysis.diffraction.xrd import XRDCalculator


# Input/output
EXPERIMENT_DIR = Path("data/exp_patterns/one_phase")
REFERENCE_DIR = Path("data/reference_structures")
OUTPUT_DIR = Path("outputs/conventional/search_match")

# Pattern and FoM settings
MIN_ANGLE = 10.0
MAX_ANGLE = 100.0
PLOT_MIN_ANGLE = 10.0
PLOT_MAX_ANGLE = 80.0
WAVELENGTH = "CuKa"
WAVELENGTH_ANGSTROM = 1.5406
MAX_EXPERIMENT_PATTERNS = None
TOP_K_TO_PRINT = 5

# Peak detection
BASELINE_PERCENTILE = 5.0
PEAK_PROMINENCE_FRACTION = 0.02
MIN_PEAK_DISTANCE_DEG = 0.22
MAX_DETECTED_PEAKS = 30

# Search-match
REFERENCE_INTENSITY_THRESHOLD = 1.0
NUM_OBS_LINES_FOR_FOM = 20
MATCH_TOLERANCE_DEG = 0.25
MIN_MATCHED_LINES_FOR_SCORE = 6

# Plot style (matching Slide-27 sizing)
FIGSIZE = (8, 7)
AXIS_LABEL_SIZE = 18
TICK_LABEL_SIZE = 15
TITLE_SIZE = 15


def normalize_0_100(y):
    y = np.asarray(y, dtype=float)
    y = y - y.min()
    return 100.0 * y / np.clip(y.max(), 1e-12, None)


def load_pattern(xy_file):
    data = np.loadtxt(xy_file)
    tt, intensity = data[:, 0], data[:, 1]
    keep = (tt >= MIN_ANGLE) & (tt <= MAX_ANGLE)
    return tt[keep], normalize_0_100(intensity[keep])


def detect_peaks(tt, intensity):
    corrected = np.clip(intensity - np.percentile(intensity, BASELINE_PERCENTILE), 0.0, None)

    step = np.median(np.diff(tt))
    min_distance_pts = max(1, int(round(MIN_PEAK_DISTANCE_DEG / step)))
    prominence = PEAK_PROMINENCE_FRACTION * (corrected.max() - corrected.min())

    peak_idx, _ = find_peaks(corrected, prominence=prominence, distance=min_distance_pts)
    if len(peak_idx) > MAX_DETECTED_PEAKS:
        keep = np.argsort(corrected[peak_idx])[::-1][:MAX_DETECTED_PEAKS]
        peak_idx = peak_idx[keep]

    peak_idx = np.sort(peak_idx)
    return peak_idx, tt[peak_idx]


def load_reference_library(cif_files):
    calc = XRDCalculator(wavelength=WAVELENGTH)
    refs = {}
    for cif in cif_files:
        pat = calc.get_pattern(Structure.from_file(cif), two_theta_range=(MIN_ANGLE, MAX_ANGLE))
        tt = np.asarray(pat.x, dtype=float)
        inten = np.asarray(pat.y, dtype=float)
        keep = inten >= REFERENCE_INTENSITY_THRESHOLD
        refs[cif.stem] = (tt[keep], inten[keep])
    return refs


def q_from_two_theta(tt):
    theta = np.deg2rad(np.asarray(tt, dtype=float) / 2.0)
    return (2.0 * np.sin(theta) / WAVELENGTH_ANGSTROM) ** 2


def greedy_match(obs_tt, ref_tt):
    candidates = []
    for i_obs, obs in enumerate(obs_tt):
        j0 = np.searchsorted(ref_tt, obs - MATCH_TOLERANCE_DEG, side="left")
        j1 = np.searchsorted(ref_tt, obs + MATCH_TOLERANCE_DEG, side="right")
        candidates.extend((abs(obs - ref_tt[j]), i_obs, j) for j in range(j0, j1))

    candidates.sort(key=lambda t: t[0])
    used_obs, used_ref, pairs = set(), set(), []
    for delta, i_obs, j_ref in candidates:
        if i_obs in used_obs or j_ref in used_ref:
            continue
        used_obs.add(i_obs)
        used_ref.add(j_ref)
        pairs.append((i_obs, j_ref, delta))
    return pairs


def score_phase(obs_peaks, ref_peaks):
    n_used = min(NUM_OBS_LINES_FOR_FOM, len(obs_peaks))
    if n_used == 0 or len(ref_peaks) == 0:
        return {
            "de_wolff": 0.0,
            "smith_snyder": 0.0,
            "n_used": 0,
            "n_match": 0,
            "n_possible": 0,
            "mean_delta_2theta": np.inf,
        }

    obs_used = obs_peaks[:n_used]
    pairs = greedy_match(obs_used, ref_peaks)

    n_possible = max(int(np.sum(ref_peaks <= obs_used[-1])), 1)
    n_match = len(pairs)

    if n_match < MIN_MATCHED_LINES_FOR_SCORE:
        return {
            "de_wolff": 0.0,
            "smith_snyder": 0.0,
            "n_used": int(n_used),
            "n_match": int(n_match),
            "n_possible": int(n_possible),
            "mean_delta_2theta": np.inf,
        }

    deltas = np.array([d for _, _, d in pairs], dtype=float)
    mean_delta = float(np.mean(deltas))
    completeness = (n_match / n_possible) * (n_match / n_used)

    # Smith-Snyder style FoM (adapted for search-match)
    smith_snyder = completeness / max(mean_delta, 1e-8)

    # de Wolff style FoM (adapted for search-match)
    q_obs = q_from_two_theta(obs_used)
    q_ref = q_from_two_theta(ref_peaks)
    q_limit = q_obs[-1]
    n20 = max(int(np.sum(q_ref <= q_limit)), 1)
    dq = np.array([abs(q_obs[i] - q_ref[j]) for i, j, _ in pairs], dtype=float)
    de_wolff = (q_limit / (2.0 * n20 * max(float(np.mean(dq)), 1e-12))) * completeness

    return {
        "de_wolff": float(de_wolff),
        "smith_snyder": float(smith_snyder),
        "n_used": int(n_used),
        "n_match": int(n_match),
        "n_possible": int(n_possible),
        "mean_delta_2theta": mean_delta,
    }


def rank_phases(obs_peaks, refs):
    rows = [{"phase": name, **score_phase(obs_peaks, tt)} for name, (tt, _) in refs.items()]
    return (
        sorted(rows, key=lambda r: r["de_wolff"], reverse=True),
        sorted(rows, key=lambda r: r["smith_snyder"], reverse=True),
    )


def print_rank_table(pattern_name, rows, score_key, label):
    rows = [r for r in rows if r[score_key] > 0.0] or rows
    print(f"\n{pattern_name}: top {TOP_K_TO_PRINT} phases by {label}")
    print("rank  phase             score     matches/used  mean |Δ2θ|")
    print("----  ----------------  --------  ------------  ----------")
    for i, row in enumerate(rows[:TOP_K_TO_PRINT], start=1):
        mean_str = "-" if not np.isfinite(row["mean_delta_2theta"]) else f"{row['mean_delta_2theta']:.3f}"
        print(
            f"{i:>4}  {row['phase']:<16}  {row[score_key]:>8.3f}  "
            f"{row['n_match']:>4}/{row['n_used']:<7}  {mean_str:>10}"
        )


def phase_title_label(phase_name):
    # Convert "Li2MnO3_15" -> "Li2MnO3 (s.g. 15)"
    formula, sg = phase_name.rsplit("_", 1)
    return f"{formula} (s.g. {sg})"


def plot_summary(pattern_name, tt, intensity, obs_peaks, best_m, best_f, refs):
    fig, axes = plt.subplots(3, 1, figsize=FIGSIZE, sharex=True)

    # Experimental profile + detected peaks
    axes[0].plot(tt, intensity, color="black", linewidth=2.2, label="Experimental")
    axes[0].scatter(obs_peaks, np.interp(obs_peaks, tt, intensity), s=30, color="#dc2626", zorder=5, label="Detected")
    axes[0].set_title(f"{pattern_name}: Experimental + detected peaks", fontsize=TITLE_SIZE, pad=4)
    axes[0].legend(fontsize=11, loc="upper right")

    # Best by de Wolff
    phase_m = best_m["phase"]
    ref_tt_m, ref_i_m = refs[phase_m]
    axes[1].fill_between(tt, 0, intensity, color="black", alpha=0.10)
    axes[1].plot(tt, intensity, color="black", linewidth=2.0)
    axes[1].vlines(ref_tt_m, 0, 0.8 * ref_i_m, color="#1f4ed8", linewidth=1.7)
    axes[1].set_title(
        f"Best by de Wolff: {phase_title_label(phase_m)}",
        fontsize=TITLE_SIZE,
        pad=4,
    )

    # Best by Smith-Snyder
    phase_f = best_f["phase"]
    ref_tt_f, ref_i_f = refs[phase_f]
    axes[2].fill_between(tt, 0, intensity, color="black", alpha=0.10)
    axes[2].plot(tt, intensity, color="black", linewidth=2.0)
    axes[2].vlines(ref_tt_f, 0, 0.8 * ref_i_f, color="#dc2626", linewidth=1.7)
    axes[2].set_title(
        f"Best by Smith-Snyder: {phase_title_label(phase_f)}",
        fontsize=TITLE_SIZE,
        pad=4,
    )

    for ax in axes:
        ax.set_xlim(PLOT_MIN_ANGLE, PLOT_MAX_ANGLE)
        ax.set_ylim(0, 105)
        ax.set_ylabel("Intensity", fontsize=AXIS_LABEL_SIZE, labelpad=12)
        ax.tick_params(axis="both", labelsize=TICK_LABEL_SIZE)

    axes[-1].set_xlabel("2θ", fontsize=AXIS_LABEL_SIZE, labelpad=12)

    out = OUTPUT_DIR / f"{pattern_name}_summary.png"
    plt.tight_layout()
    plt.savefig(out, dpi=200)
    plt.close(fig)
    print(f"  Saved plot: {out}")


def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    exp_files = sorted(EXPERIMENT_DIR.glob("*.xy"))
    if MAX_EXPERIMENT_PATTERNS is not None:
        exp_files = exp_files[:MAX_EXPERIMENT_PATTERNS]

    refs = load_reference_library(sorted(REFERENCE_DIR.glob("*.cif")))
    all_rows = []

    print("\n=== Peak Search-Match Demo (de Wolff + Smith-Snyder) ===")
    print(f"Experimental patterns: {len(exp_files)}")
    print(f"Reference phases:      {len(refs)}")

    for exp_file in exp_files:
        name = exp_file.stem
        tt, intensity = load_pattern(exp_file)
        _, obs_peaks = detect_peaks(tt, intensity)

        print(f"\n--- {name} ---")
        print(f"Detected peaks: {len(obs_peaks)}")

        by_m, by_f = rank_phases(obs_peaks, refs)
        print_rank_table(name, by_m, "de_wolff", "de Wolff")
        print_rank_table(name, by_f, "smith_snyder", "Smith-Snyder")

        plot_summary(name, tt, intensity, obs_peaks, by_m[0], by_f[0], refs)

        for row in by_m:
            all_rows.append({"pattern": name, "phase": row["phase"], **{k: row[k] for k in row if k != "phase"}})

    csv_file = OUTPUT_DIR / "all_pattern_rankings.csv"
    with open(csv_file, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "pattern",
                "phase",
                "de_wolff",
                "smith_snyder",
                "n_used",
                "n_match",
                "n_possible",
                "mean_delta_2theta",
            ],
        )
        writer.writeheader()
        writer.writerows(all_rows)

    print(f"\nSaved ranking table: {csv_file}")


# 02a — Search & Match (Peak Lists)

We detect peaks in experimental patterns and rank candidate phases using FoM-style line matching scores.

## Run the Demo
The next cell executes the shared tutorial module.

In [ ]:
TOP_K_TO_PRINT = 3
MAX_EXPERIMENT_PATTERNS = 4
main()


## What To Observe
Compare top-ranked phases and where the reference sticks align with experimental peaks.

In [ ]:
for p in sorted(glob.glob("outputs/conventional/search_match/*_summary.png"))[:3]:
    display(Image(p))

## Summary
- Peak-list matching is fast and interpretable.
- Performance depends on peak detection quality and tolerance settings.
- This method can struggle when peaks overlap or broaden heavily.

## Next Steps
Continue to the next section below in this notebook.

## 02b - Profile Correlation

In [ ]:
import glob
from IPython.display import Image, display

# Inline tutorial script
from pathlib import Path
import csv

# For handling arrays
import numpy as np

# For plotting
import matplotlib.pyplot as plt

# To load structures and compute XRD stick patterns
from pymatgen.core import Structure
from pymatgen.analysis.diffraction.xrd import XRDCalculator


# Input/output
EXPERIMENT_DIR = Path("data/exp_patterns/one_phase")
REFERENCE_DIR = Path("data/reference_structures")
OUTPUT_DIR = Path("outputs/conventional/profile_correlation")

# Pattern and simulation settings
MIN_ANGLE = 10.0
MAX_ANGLE = 80.0
WAVELENGTH = "CuKa"
REFERENCE_INTENSITY_THRESHOLD = 1.0

# Continuous-profile broadening (same spirit as Slide-21)
FWHM = 0.30
GAUSS_FRAC = 0.2

# Experimental preprocessing
BASELINE_PERCENTILE = 5.0

# Reporting
MAX_EXPERIMENT_PATTERNS = None
TOP_K_TO_PRINT = 5

# Plot style (matching Slide-27 sizing)
FIGSIZE = (8, 7)
AXIS_LABEL_SIZE = 18
TICK_LABEL_SIZE = 15
TITLE_SIZE = 15


def normalize_0_100(y):
    y = np.asarray(y, dtype=float)
    y = y - y.min()
    return 100.0 * y / np.clip(y.max(), 1e-12, None)


def phase_title_label(phase_name):
    # Convert "Li2MnO3_15" -> "Li2MnO3 (s.g. 15)"
    formula, sg = phase_name.rsplit("_", 1)
    return f"{formula} (s.g. {sg})"


def load_experimental_profile(xy_file):
    data = np.loadtxt(xy_file)
    two_theta = data[:, 0]
    intensity = data[:, 1]

    keep = (two_theta >= MIN_ANGLE) & (two_theta <= MAX_ANGLE)
    two_theta = two_theta[keep]
    intensity = intensity[keep]

    # Simple baseline removal so correlations focus on pattern shape.
    intensity = np.clip(intensity - np.percentile(intensity, BASELINE_PERCENTILE), 0.0, None)
    intensity = normalize_0_100(intensity)

    return two_theta, intensity


def load_reference_stick_library(cif_files):
    calculator = XRDCalculator(wavelength=WAVELENGTH)
    refs = {}

    for cif_file in cif_files:
        pattern = calculator.get_pattern(
            Structure.from_file(cif_file),
            two_theta_range=(MIN_ANGLE, MAX_ANGLE),
        )
        peak_pos = np.asarray(pattern.x, dtype=float)
        peak_intensity = np.asarray(pattern.y, dtype=float)

        keep = peak_intensity >= REFERENCE_INTENSITY_THRESHOLD
        peak_pos = peak_pos[keep]
        peak_intensity = normalize_0_100(peak_intensity[keep])

        refs[cif_file.stem] = (peak_pos, peak_intensity)

    return refs


def simulate_continuous_profile(two_theta_grid, peak_pos, peak_intensity):
    if len(peak_pos) == 0:
        return np.zeros_like(two_theta_grid)

    dx = two_theta_grid[:, None] - peak_pos[None, :]

    sigma = FWHM / (2.0 * np.sqrt(2.0 * np.log(2.0)))
    gamma = FWHM / 2.0

    gauss = np.exp(-0.5 * (dx / sigma) ** 2)
    lorentz = (gamma**2) / (dx**2 + gamma**2)

    profile = ((1.0 - GAUSS_FRAC) * gauss + GAUSS_FRAC * lorentz) @ peak_intensity
    return normalize_0_100(profile)


def pearson_corr(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return 0.0
    return float(np.corrcoef(a, b)[0, 1])


def cosine_similarity(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom < 1e-12:
        return 0.0
    return float(np.dot(a, b) / denom)


def rank_phases(exp_profile, exp_two_theta, reference_library):
    rows = []
    simulated_profiles = {}

    for phase, (peak_pos, peak_intensity) in reference_library.items():
        sim_profile = simulate_continuous_profile(exp_two_theta, peak_pos, peak_intensity)

        rows.append(
            {
                "phase": phase,
                "pearson": pearson_corr(exp_profile, sim_profile),
                "cosine": cosine_similarity(exp_profile, sim_profile),
            }
        )
        simulated_profiles[phase] = sim_profile

    by_pearson = sorted(rows, key=lambda r: r["pearson"], reverse=True)
    by_cosine = sorted(rows, key=lambda r: r["cosine"], reverse=True)
    return by_pearson, by_cosine, simulated_profiles


def print_rank_table(pattern_name, rows, score_key, label):
    print(f"\n{pattern_name}: top {TOP_K_TO_PRINT} phases by {label}")
    print("rank  phase             score")
    print("----  ----------------  ------")
    for i, row in enumerate(rows[:TOP_K_TO_PRINT], start=1):
        print(f"{i:>4}  {row['phase']:<16}  {row[score_key]:>6.3f}")


def plot_summary(pattern_name, two_theta, exp_profile, best_pearson, best_cosine, simulated_profiles):
    fig, axes = plt.subplots(nrows=3, ncols=1, figsize=FIGSIZE, sharex=True)

    # 1) Experimental profile
    axes[0].fill_between(two_theta, 0, exp_profile, color="black", alpha=0.15)
    axes[0].plot(two_theta, exp_profile, color="black", linewidth=2.2)
    axes[0].set_title(f"{pattern_name}: Experimental profile", fontsize=TITLE_SIZE, pad=4)

    # 2) Best by Pearson
    pearson_phase = best_pearson["phase"]
    axes[1].fill_between(two_theta, 0, exp_profile, color="black", alpha=0.10)
    axes[1].plot(two_theta, exp_profile, color="black", linewidth=2.0, label="Experimental")
    axes[1].plot(two_theta, simulated_profiles[pearson_phase], color="#1f4ed8", linewidth=2.0, label="Simulated")
    axes[1].set_title(f"Best by Pearson: {phase_title_label(pearson_phase)}", fontsize=TITLE_SIZE, pad=4)
    axes[1].legend(fontsize=11, loc="upper right")

    # 3) Best by Cosine
    cosine_phase = best_cosine["phase"]
    axes[2].fill_between(two_theta, 0, exp_profile, color="black", alpha=0.10)
    axes[2].plot(two_theta, exp_profile, color="black", linewidth=2.0, label="Experimental")
    axes[2].plot(two_theta, simulated_profiles[cosine_phase], color="#dc2626", linewidth=2.0, label="Simulated")
    axes[2].set_title(f"Best by Cosine: {phase_title_label(cosine_phase)}", fontsize=TITLE_SIZE, pad=4)
    axes[2].legend(fontsize=11, loc="upper right")

    for ax in axes:
        ax.set_xlim(MIN_ANGLE, MAX_ANGLE)
        ax.set_ylim(0, 105)
        ax.set_ylabel("Intensity", fontsize=AXIS_LABEL_SIZE, labelpad=12)
        ax.tick_params(axis="both", labelsize=TICK_LABEL_SIZE)

    axes[-1].set_xlabel("2θ", fontsize=AXIS_LABEL_SIZE, labelpad=12)

    out_file = OUTPUT_DIR / f"{pattern_name}_profile-correlation.png"
    plt.tight_layout()
    plt.savefig(out_file, dpi=200)
    plt.close(fig)
    print(f"  Saved plot: {out_file}")


def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    exp_files = sorted(EXPERIMENT_DIR.glob("*.xy"))
    if MAX_EXPERIMENT_PATTERNS is not None:
        exp_files = exp_files[:MAX_EXPERIMENT_PATTERNS]

    reference_library = load_reference_stick_library(sorted(REFERENCE_DIR.glob("*.cif")))

    all_rows = []
    print("\n=== Full-Profile Correlation Demo (Pearson + Cosine) ===")
    print(f"Experimental patterns: {len(exp_files)}")
    print(f"Reference phases:      {len(reference_library)}")

    for exp_file in exp_files:
        pattern_name = exp_file.stem
        two_theta, exp_profile = load_experimental_profile(exp_file)

        by_pearson, by_cosine, simulated_profiles = rank_phases(
            exp_profile,
            two_theta,
            reference_library,
        )

        print(f"\n--- {pattern_name} ---")
        print_rank_table(pattern_name, by_pearson, "pearson", "Pearson")
        print_rank_table(pattern_name, by_cosine, "cosine", "Cosine")

        plot_summary(
            pattern_name,
            two_theta,
            exp_profile,
            by_pearson[0],
            by_cosine[0],
            simulated_profiles,
        )

        for row in by_pearson:
            all_rows.append(
                {
                    "pattern": pattern_name,
                    "phase": row["phase"],
                    "pearson": row["pearson"],
                    "cosine": row["cosine"],
                }
            )

    csv_file = OUTPUT_DIR / "all_pattern_profile-correlations.csv"
    with open(csv_file, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["pattern", "phase", "pearson", "cosine"])
        writer.writeheader()
        writer.writerows(all_rows)

    print(f"\nSaved ranking table: {csv_file}")


# 02b — Full-Profile Correlation

Instead of line matching, this method compares full simulated and observed profiles via correlation metrics.

## Run the Demo
The next cell executes the shared tutorial module.

In [ ]:
TOP_K_TO_PRINT = 3
MAX_EXPERIMENT_PATTERNS = 4
main()


## What To Observe
Look for differences between Pearson and cosine rankings on the same pattern.

In [ ]:
for p in sorted(glob.glob("outputs/conventional/profile_correlation/*_profile-correlation.png"))[:3]:
    display(Image(p))

## Summary
- Profile-level comparison uses more information than discrete peak lists.
- Pearson and cosine can prioritize slightly different candidates.
- Baseline handling strongly influences correlation scores.

## Next Steps
Continue to the next section below in this notebook.

## 02c - Rietveld Refinement

In [ ]:
import glob
from IPython.display import Image, display

# Inline tutorial script
from pathlib import Path
import csv

# For handling arrays
import numpy as np

# For plotting
import matplotlib.pyplot as plt

# For optimization
from scipy.optimize import minimize

# To load structures and compute XRD stick patterns
from pymatgen.core import Structure
from pymatgen.analysis.diffraction.xrd import XRDCalculator


# Input/output
EXPERIMENT_DIR = Path("data/exp_patterns/one_phase")
REFERENCE_DIR = Path("data/reference_structures")
OUTPUT_DIR = Path("outputs/conventional/rietveld")

# Pattern settings
MIN_ANGLE = 10.0
MAX_ANGLE = 80.0
WAVELENGTH = "CuKa"
REFERENCE_INTENSITY_THRESHOLD = 1.0

# Refinement settings
BACKGROUND_DEGREE = 6  # higher degree -> more flexible/curvy background
FWHM_INIT = 0.30
GAUSS_FRAC = 0.2

LATTICE_SCALE_BOUNDS = (0.98, 1.02)  # refine a,b,c scale factors
FWHM_BOUNDS = (0.05, 1.20)

LATTICE_MAXITER = 40
WIDTH_MAXITER = 40

# Reporting
MAX_EXPERIMENT_PATTERNS = None
TOP_K_TO_PRINT = 5
PATTERNS_TO_RUN = ["TiO2"]  # set to None to run all patterns

# Plot style (matching Slide-27 sizing)
FIGSIZE = (8, 7)
AXIS_LABEL_SIZE = 18
TICK_LABEL_SIZE = 15
TITLE_SIZE = 15


def normalize_0_100(y):
    y = np.asarray(y, dtype=float)
    y = y - y.min()
    return 100.0 * y / np.clip(y.max(), 1e-12, None)


def phase_title_label(phase_name):
    # Convert "Li2MnO3_15" -> "Li2MnO3 (s.g. 15)"
    formula, sg = phase_name.rsplit("_", 1)
    return f"{formula} (s.g. {sg})"


def load_experimental_profile(xy_file):
    data = np.loadtxt(xy_file)
    two_theta = data[:, 0]
    intensity = data[:, 1]

    keep = (two_theta >= MIN_ANGLE) & (two_theta <= MAX_ANGLE)
    two_theta = two_theta[keep]
    intensity = intensity[keep]

    # Keep observed profile scale simple and consistent.
    intensity = normalize_0_100(intensity)
    return two_theta, intensity


def load_reference_structures(cif_files):
    return {cif.stem: Structure.from_file(cif) for cif in cif_files}


def apply_lattice_scales(structure, scales):
    s = structure.copy()
    # scale = 1.00 means no change; strain is relative (scale - 1).
    s.apply_strain([scales[0] - 1.0, scales[1] - 1.0, scales[2] - 1.0])
    return s


def get_stick_pattern(structure, calculator):
    pattern = calculator.get_pattern(structure, two_theta_range=(MIN_ANGLE, MAX_ANGLE))
    peak_pos = np.asarray(pattern.x, dtype=float)
    peak_intensity = np.asarray(pattern.y, dtype=float)

    keep = peak_intensity >= REFERENCE_INTENSITY_THRESHOLD
    peak_pos = peak_pos[keep]
    peak_intensity = normalize_0_100(peak_intensity[keep])
    return peak_pos, peak_intensity


def simulate_profile(two_theta_grid, peak_pos, peak_intensity, fwhm):
    if len(peak_pos) == 0:
        return np.zeros_like(two_theta_grid)

    dx = two_theta_grid[:, None] - peak_pos[None, :]

    sigma = fwhm / (2.0 * np.sqrt(2.0 * np.log(2.0)))
    gamma = fwhm / 2.0

    gauss = np.exp(-0.5 * (dx / sigma) ** 2)
    lorentz = (gamma**2) / (dx**2 + gamma**2)

    profile = ((1.0 - GAUSS_FRAC) * gauss + GAUSS_FRAC * lorentz) @ peak_intensity
    return normalize_0_100(profile)


def fit_scale_only(y_obs, y_phase, y_bg):
    # Solve y_obs ≈ scale * y_phase + y_bg for scale (least squares, clipped positive).
    numer = np.dot(y_phase, (y_obs - y_bg))
    denom = np.dot(y_phase, y_phase)
    scale = 0.0 if denom < 1e-12 else max(0.0, numer / denom)
    y_fit = scale * y_phase + y_bg
    return scale, y_fit


def fit_background_and_scale(y_obs, y_phase, x_scaled, degree):
    # Linear model: y ≈ scale * phase + Chebyshev background.
    cheb_basis = np.polynomial.chebyshev.chebvander(x_scaled, degree)
    A = np.column_stack([y_phase, cheb_basis])
    params, *_ = np.linalg.lstsq(A, y_obs, rcond=None)

    scale = max(0.0, float(params[0]))
    bg_coeffs = np.asarray(params[1:], dtype=float)

    y_bg = np.polynomial.chebyshev.chebval(x_scaled, bg_coeffs)
    y_fit = scale * y_phase + y_bg
    return scale, bg_coeffs, y_bg, y_fit


def compute_rwp(y_obs, y_calc):
    # Simple weighted profile R-factor.
    w = 1.0 / np.clip(y_obs, 1e-3, None)
    numer = np.sum(w * (y_obs - y_calc) ** 2)
    denom = np.sum(w * y_obs**2)
    return 100.0 * np.sqrt(numer / np.clip(denom, 1e-12, None))


def pearson_corr(a, b):
    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return 0.0
    return float(np.corrcoef(a, b)[0, 1])


def refine_phase_sequential(two_theta, y_obs, base_structure, calculator):
    x_scaled = 2.0 * (two_theta - MIN_ANGLE) / (MAX_ANGLE - MIN_ANGLE) - 1.0

    # ---------------------------------------------------------------------
    # Step 1: refine background (polynomial), with initial lattice/width.
    # ---------------------------------------------------------------------
    peak_pos_0, peak_int_0 = get_stick_pattern(base_structure, calculator)
    y_phase_0 = simulate_profile(two_theta, peak_pos_0, peak_int_0, FWHM_INIT)
    _, bg_coeffs, y_bg, y_fit_1 = fit_background_and_scale(y_obs, y_phase_0, x_scaled, BACKGROUND_DEGREE)

    # ---------------------------------------------------------------------
    # Step 2: refine lattice parameters (a,b,c scales), background fixed.
    # ---------------------------------------------------------------------
    def lattice_objective(scales):
        s = np.clip(np.asarray(scales, dtype=float), *LATTICE_SCALE_BOUNDS)
        refined_structure = apply_lattice_scales(base_structure, s)
        peak_pos, peak_int = get_stick_pattern(refined_structure, calculator)
        y_phase = simulate_profile(two_theta, peak_pos, peak_int, FWHM_INIT)
        _, y_fit = fit_scale_only(y_obs, y_phase, y_bg)
        return np.mean((y_obs - y_fit) ** 2)

    res_lat = minimize(
        lattice_objective,
        x0=np.array([1.0, 1.0, 1.0]),
        method="Powell",
        bounds=[LATTICE_SCALE_BOUNDS, LATTICE_SCALE_BOUNDS, LATTICE_SCALE_BOUNDS],
        options={"maxiter": LATTICE_MAXITER, "xtol": 1e-3, "ftol": 1e-3},
    )
    best_scales = np.clip(np.asarray(res_lat.x, dtype=float), *LATTICE_SCALE_BOUNDS)

    structure_2 = apply_lattice_scales(base_structure, best_scales)
    peak_pos_2, peak_int_2 = get_stick_pattern(structure_2, calculator)
    y_phase_2 = simulate_profile(two_theta, peak_pos_2, peak_int_2, FWHM_INIT)
    _, y_fit_2 = fit_scale_only(y_obs, y_phase_2, y_bg)

    # ---------------------------------------------------------------------
    # Step 3: refine peak width (FWHM), lattice/background fixed.
    # ---------------------------------------------------------------------
    def width_objective(width):
        w = float(np.clip(width[0], *FWHM_BOUNDS))
        y_phase = simulate_profile(two_theta, peak_pos_2, peak_int_2, w)
        _, y_fit = fit_scale_only(y_obs, y_phase, y_bg)
        return np.mean((y_obs - y_fit) ** 2)

    res_w = minimize(
        width_objective,
        x0=np.array([FWHM_INIT]),
        method="Powell",
        bounds=[FWHM_BOUNDS],
        options={"maxiter": WIDTH_MAXITER, "xtol": 1e-3, "ftol": 1e-3},
    )
    best_fwhm = float(np.clip(res_w.x[0], *FWHM_BOUNDS))

    y_phase_3 = simulate_profile(two_theta, peak_pos_2, peak_int_2, best_fwhm)
    scale_3, y_fit_3 = fit_scale_only(y_obs, y_phase_3, y_bg)

    return {
        "bg_coeffs": bg_coeffs,
        "scales": best_scales,
        "fwhm": best_fwhm,
        "scale": scale_3,
        "y_bg": y_bg,
        "y_fit_step1": y_fit_1,
        "y_fit_step2": y_fit_2,
        "y_fit_final": y_fit_3,
        "rwp": compute_rwp(y_obs, y_fit_3),
        "pearson": pearson_corr(y_obs, y_fit_3),
    }


def print_rank_table(pattern_name, rows):
    print(f"\n{pattern_name}: top {TOP_K_TO_PRINT} phases by final Rwp (lower is better)")
    print("rank  phase             Rwp(%)  Pearson   a_scale  b_scale  c_scale  FWHM")
    print("----  ----------------  ------  -------  -------  -------  -------  -----")
    for i, row in enumerate(rows[:TOP_K_TO_PRINT], start=1):
        s = row["scales"]
        print(
            f"{i:>4}  {row['phase']:<16}  {row['rwp']:>6.2f}  {row['pearson']:>7.3f}  "
            f"{s[0]:>7.4f}  {s[1]:>7.4f}  {s[2]:>7.4f}  {row['fwhm']:>5.3f}"
        )


def plot_refinement_summary(pattern_name, two_theta, y_obs, best_row):
    fig, axes = plt.subplots(nrows=3, ncols=1, figsize=FIGSIZE, sharex=True)

    # Step 1: background refinement
    axes[0].plot(two_theta, y_obs, color="black", linewidth=2.0, label="Experimental")
    axes[0].plot(two_theta, best_row["y_bg"], color="#6b7280", linewidth=1.8, label="Background")
    axes[0].plot(two_theta, best_row["y_fit_step1"], color="#1f4ed8", linewidth=1.8, label="Step 1 fit")
    axes[0].set_title("Step 1: Background refinement", fontsize=TITLE_SIZE, pad=4)
    axes[0].legend(fontsize=10, loc="upper right")

    # Step 2: lattice refinement
    axes[1].plot(two_theta, y_obs, color="black", linewidth=2.0, label="Experimental")
    axes[1].plot(two_theta, best_row["y_fit_step2"], color="#1f4ed8", linewidth=1.8, label="Step 2 fit")
    axes[1].set_title("Step 2: Lattice-parameter refinement", fontsize=TITLE_SIZE, pad=4)
    axes[1].legend(fontsize=10, loc="upper right")

    # Step 3: peak-width refinement (final)
    axes[2].plot(two_theta, y_obs, color="black", linewidth=2.0, label="Experimental")
    axes[2].plot(two_theta, best_row["y_fit_final"], color="#dc2626", linewidth=1.9, label="Final fit")
    axes[2].set_title(f"Step 3: Peak-width refinement ({phase_title_label(best_row['phase'])})", fontsize=TITLE_SIZE, pad=4)
    axes[2].legend(fontsize=10, loc="upper right")

    for ax in axes:
        ax.set_xlim(MIN_ANGLE, MAX_ANGLE)
        ax.set_ylim(0, 105)
        ax.set_ylabel("Intensity", fontsize=AXIS_LABEL_SIZE, labelpad=12)
        ax.tick_params(axis="both", labelsize=TICK_LABEL_SIZE)

    axes[-1].set_xlabel("2θ", fontsize=AXIS_LABEL_SIZE, labelpad=12)

    out_file = OUTPUT_DIR / f"{pattern_name}_rietveld-sequential.png"
    plt.tight_layout()
    plt.savefig(out_file, dpi=200)
    plt.close(fig)
    print(f"  Saved plot: {out_file}")


def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    exp_files = sorted(EXPERIMENT_DIR.glob("*.xy"))
    if PATTERNS_TO_RUN is not None:
        allowed = set(PATTERNS_TO_RUN)
        exp_files = [f for f in exp_files if f.stem in allowed]
    if MAX_EXPERIMENT_PATTERNS is not None:
        exp_files = exp_files[:MAX_EXPERIMENT_PATTERNS]

    structures = load_reference_structures(sorted(REFERENCE_DIR.glob("*.cif")))
    calculator = XRDCalculator(wavelength=WAVELENGTH)

    all_rows = []
    print("\n=== Sequential Rietveld-Style Refinement Demo ===")
    print("Refinement order: (1) background -> (2) lattice params -> (3) peak width")
    print(f"Experimental patterns: {len(exp_files)}")
    print(f"Reference phases:      {len(structures)}")

    for exp_file in exp_files:
        pattern_name = exp_file.stem
        two_theta, y_obs = load_experimental_profile(exp_file)

        rows = []
        for phase, structure in structures.items():
            result = refine_phase_sequential(two_theta, y_obs, structure, calculator)
            rows.append({"phase": phase, **result})

        rows = sorted(rows, key=lambda r: r["rwp"])
        best_row = rows[0]

        print(f"\n--- {pattern_name} ---")
        print_rank_table(pattern_name, rows)
        plot_refinement_summary(pattern_name, two_theta, y_obs, best_row)

        for row in rows:
            all_rows.append(
                {
                    "pattern": pattern_name,
                    "phase": row["phase"],
                    "rwp": row["rwp"],
                    "pearson": row["pearson"],
                    "a_scale": row["scales"][0],
                    "b_scale": row["scales"][1],
                    "c_scale": row["scales"][2],
                    "fwhm": row["fwhm"],
                }
            )

    csv_file = OUTPUT_DIR / "all_pattern_rietveld-rankings.csv"
    with open(csv_file, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["pattern", "phase", "rwp", "pearson", "a_scale", "b_scale", "c_scale", "fwhm"],
        )
        writer.writeheader()
        writer.writerows(all_rows)

    print(f"\nSaved ranking table: {csv_file}")


# 02c — Sequential Rietveld-Style Refinement

This simplified workflow refines background, lattice scales, and width parameters in sequence for each candidate phase.

## Run the Demo
The next cell executes the shared tutorial module.

In [ ]:
TOP_K_TO_PRINT = 3
PATTERNS_TO_RUN = ["TiO2", "ZrO2"]
main()


## What To Observe
Track how each refinement stage improves the fit and lowers Rwp.

In [ ]:
for p in sorted(glob.glob("outputs/conventional/rietveld/*_rietveld-sequential.png")):
    display(Image(p))

## Summary
- Sequential refinement isolates effects of key parameter groups.
- Rwp provides a compact fit-quality ranking.
- Refinement-based methods are accurate but more compute-intensive.

## Next Steps
Continue to **03 — ML + Deep Learning**: [Open in Colab](https://colab.research.google.com/github/Szymanski-Group/MRS_CH08_Tutorial/blob/main/notebooks/03_ML-Deep-Learning.ipynb)